# Notebook 01 — SD2.1 + cgDDI acne textual inversion + optional facial-acne LoRA + Compel
**Kaggle inference notebook.** Five images per morphology prompt. Long prompts use **Compel 2.3.1**, with `truncate_long_prompts=False` and a strict maximum of **150 CLIP content tokens** for each positive and negative prompt. On SD2.1, a 77-token CLIP window includes two special tokens, leaving at most 75 content tokens per chunk (two chunks = 150 content tokens; embedding sequence length up to 154). Prompts are never silently shortened.

Uses the official `hcarrion/acne-cystic` **textual inversion**, **not** a LoRA from that repository. The additional facial-acne LoRA is optional and comes from Notebook 02. Model loading and GPU inference require Kaggle T4 and Internet access; sandbox tests are CPU-only.

**Run logs:** every generation attempt saves `notebook_console.log` (including tokenizer or CUDA errors) and `run_log.jsonl`. Successful outputs include per-image SHA-256 manifest, human-review draft CSV and summary JSON. Preserve all artifacts in Kaggle outputs; only privacy-reviewed aggregate statistics should be committed to public GitHub.


## 1. Setup and versions
Kaggle sidebar → **Settings** → GPU **T4** (one GPU suffices), Internet ON. Run in a fresh kernel. Do not install/reinstall PyTorch; use Kaggle's CUDA build.

In [ ]:
# Versions pinned to one compatible Diffusers/Transformers/Compel stack.
# In Kaggle use a fresh session with GPU enabled and Internet ON.
# Leave Kaggle's CUDA-enabled PyTorch intact.
import sys, subprocess, importlib.metadata as metadata
PACKAGES = [
    "diffusers==0.35.1", "transformers==4.56.2", "accelerate==1.10.1",
    "peft==0.17.1", "datasets==3.6.0", "huggingface_hub==0.34.4",
    "safetensors==0.6.2", "ftfy==6.3.1", "imagehash==4.3.2",
    "tensorboard==2.20.0", "compel==2.3.1",
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
for spec in PACKAGES:
    package, expected = spec.split('==')
    got = metadata.version(package)
    if got != expected:
        raise RuntimeError(f'Package mismatch: {package} {got}; expected {expected}. Restart Kaggle session then rerun.')
print('Pinned packages installed. Use a fresh kernel if packages had been imported previously.')


In [ ]:
import os, json, csv, time, hashlib, sys, math
from pathlib import Path
import torch, diffusers, transformers, accelerate, compel as compel_package
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from compel import Compel, DiffusersTextualInversionManager, SplitLongTextMode
from PIL import Image, ImageOps
from IPython.display import display

assert torch.cuda.is_available(), 'GPU missing: Kaggle Settings → Accelerator → GPU T4.'
assert diffusers.__version__ == '0.35.1', 'Restart kernel: expected Diffusers 0.35.1.'
assert transformers.__version__ == '4.56.2', 'Restart kernel: expected Transformers 4.56.2.'
from importlib.metadata import version as _installed_version
assert _installed_version('compel') == '2.3.1', 'Restart kernel: expected Compel 2.3.1.'
print('torch',torch.__version__, 'diffusers',diffusers.__version__,
      'transformers',transformers.__version__, 'compel',_installed_version('compel'))
print('GPU:',torch.cuda.get_device_name(0), 'CUDA:',torch.version.cuda)


## 2. Configuration
**EDIT HERE** for the base, output directory, and optional trained LoRA. `LORA_SOURCE` accepts the *directory* holding `pytorch_lora_weights.safetensors`, a `.safetensors` path, or a compatible Hub model repo. Leave blank for baseline A. Set it to `/kaggle/input/your-acne-lora/...` after exporting Notebook 02 weights. Use a verified SD2.1-compatible base; do **not** use an SD1.5 or SDXL LoRA with this base.
**Long prompts:** `MAX_CONTENT_TOKENS=150` is a hard acceptance limit, not a request to pad your prompt to 150. This includes the optional `<acne-cystic-class>` token and checks both positive and negative prompts.

In [ ]:
BASE_MODEL = "sd2-community/stable-diffusion-2-1-base"  # unofficial community mirror of SD2.1 512 base
TI_MODEL = "hcarrion/acne-cystic"
LORA_SOURCE = ""   # e.g. "/kaggle/input/our-acne-lora" or "/kaggle/working/acne_lora_output"
LORA_SCALE = 0.8    # try 0.6 / 0.8 / 1.0; not clinically optimized
OUT_DIR = Path('/kaggle/working/acne_generated')
IMAGES_PER_PROMPT = 5
MAX_CONTENT_TOKENS = 150   # strict upper limit for both positive and negative prompts
EXPECTED_CLIP_WINDOW = 77
MAX_CLIP_CHUNKS = 2
HEIGHT = WIDTH = 512
STEPS = 30
GUIDANCE = 7.0
BASE_SEED = 34021
NEGATIVE_PROMPT = 'cartoon, illustration, CGI, painting, airbrushed skin, plastic skin, excessive smoothing, blurry, deformed face, duplicate face, text, watermark'
assert IMAGES_PER_PROMPT >= 1 and HEIGHT % 8 == 0 and WIDTH % 8 == 0
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:',OUT_DIR)

## 3. Authored morphology prompts
Uses the project `prompts/` JSON files, 300 per class with protected group-disjoint train/validation/test prompt families. Defaults to a five-prompt **train-only** pilot and uses each prompt's original negative text. Change `PROMPT_LIMIT=None` to run all 210 train prompts. Outputs include a review-draft CSV; all generated images begin **unapproved**.


In [ ]:
# Split-safe authored prompts (210 train, 45 val, 45 test; 60 disjoint families).
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prompt_collection,generation_manifest,append_run_log
PROMPT_PARTITION='train'
PROMPT_VARIANT='morphology'  # use 'generic' for E1, morphology for E2  # DO NOT augment with val/test prompt families
PROMPT_OFFSET=0
PROMPT_LIMIT=5  # Kaggle pilot; None for all 210 training prompts
PROMPTS=prompt_collection(DERMSYNREAL_REPO,'Acne',partition=PROMPT_PARTITION,limit=PROMPT_LIMIT,offset=PROMPT_OFFSET,variant=PROMPT_VARIANT)
# Keep E1 generic and E2 morphology images in separate run directories; never overwrite.
OUT_DIR=Path(OUT_DIR)/('prompt_'+PROMPT_VARIANT)
if OUT_DIR.exists():
    raise FileExistsError(f'Generation output already exists: {OUT_DIR}; start a fresh run directory')
print('Authored prompts:',len(PROMPTS),'families:',len({p['prompt_family_id'] for p in PROMPTS}))

In [ ]:
# Capture generation console, warnings and failed diffusion execution to disk.
from dermsynreal.kaggle_runs import start_notebook_logging
GENERATION_CONSOLE=start_notebook_logging(OUT_DIR,kind='diffusion_generation',
    config={'prompt_partition':PROMPT_PARTITION,'prompt_variant':PROMPT_VARIANT,
            'prompt_count':len(PROMPTS),'class_name':'Acne',
            'images_per_prompt':IMAGES_PER_PROMPT},repo_root=DERMSYNREAL_REPO)
print('Generation console file:',GENERATION_CONSOLE)


## 4. Load base and two optional adaptations
This model card documents `pipe.load_textual_inversion("hcarrion/acne-cystic")` and its `<acne-cystic-class>` token. The local LoRA is a separate weight file created by the training notebook; it is **not** downloaded from `hcarrion/acne-cystic`.

The SD2.1 model can be several GB. If Hugging Face asks for authentication/licence consent, follow its instructions; if weights are unavailable, do not silently switch architecture.
**Compel is initialized only after** both the textual inversion and optional LoRA are loaded and the pipeline is on CUDA, so its embedding provider uses the final tokenizer and text encoder.

In [ ]:
from huggingface_hub import model_info
# Record source revisions for reproducibility. Network must be enabled on the first run.
base_commit = model_info(BASE_MODEL).sha
adapter_commit = model_info(TI_MODEL).sha
print('Base revision:', base_commit, 'TI revision:', adapter_commit)
pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, revision=base_commit, torch_dtype=torch.float16, use_safetensors=True,
)
# Use the published embedding, not a fictitious LoRA from its model card.
pipe.load_textual_inversion(TI_MODEL, revision=adapter_commit)
assert '<acne-cystic-class>' in pipe.tokenizer.get_vocab(), 'Published acne TI token was not registered.'

if LORA_SOURCE.strip():
    source = str(LORA_SOURCE).strip()
    if Path(source).is_file():
        assert source.endswith('.safetensors'), 'Pass a .safetensors LoRA file or a directory.'
        pipe.load_lora_weights(str(Path(source).parent), weight_name=Path(source).name, adapter_name='facial_acne')
    else:
        pipe.load_lora_weights(source, adapter_name='facial_acne')
    pipe.set_adapters('facial_acne', adapter_weights=LORA_SCALE)
    print('Custom LoRA activated:', source, 'scale',LORA_SCALE)
else:
    print('Mode A: base + published acne textual inversion only (no custom LoRA).')

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.to('cuda')
pipe.set_progress_bar_config(disable=False)
print('Pipeline ready: base + textual inversion' + (' + custom LoRA' if LORA_SOURCE else ''))

## 5. Compel-based two-chunk prompt preparation
SD2.1 CLIP's 77-position input contains 75 usable content token positions plus its BOS/EOS tokens. **150 content tokens fit in at most two CLIP chunks**, each independently wrapped in special tokens. The notebook validates both prompts with `pipe.tokenizer` using `add_special_tokens=False`, and checks the effective textual-inversion-expanded form. Over-budget prompts raise `ValueError` **before any image is generated**. Compel encodes positive and negative prompts with `truncate_long_prompts=False`, then pads both tensors to the same sequence length. It also checks the embeddings are no longer than two chunks.

These are token counts rather than English word counts. A two-chunk maximum does not guarantee every length or weighting pattern uses exactly two chunks. The notebook deliberately fails if Compel needs more than two.

In [ ]:
# The diffusion pipeline is already loaded with its registered textual inversion.
CLIP_WINDOW = int(pipe.tokenizer.model_max_length)
assert CLIP_WINDOW == EXPECTED_CLIP_WINDOW, (
    f'Expected SD2.1 CLIP window 77, found {CLIP_WINDOW}. Check BASE_MODEL.')
SPECIAL_TOKENS = int(pipe.tokenizer.num_special_tokens_to_add(pair=False))
CONTENT_PER_CHUNK = CLIP_WINDOW - SPECIAL_TOKENS
assert CONTENT_PER_CHUNK == 75, (
    f'Expected 75 content tokens per SD2.1 chunk, found {CONTENT_PER_CHUNK}.')
assert MAX_CONTENT_TOKENS == 150 and MAX_CLIP_CHUNKS == 2
assert MAX_CONTENT_TOKENS <= MAX_CLIP_CHUNKS * CONTENT_PER_CHUNK

def prompt_text(entry):
    text = str(entry['text']).strip()
    if entry.get('use_cystic_token', False) and '<acne-cystic-class>' not in text:
        text += ', <acne-cystic-class>'
    return text

def count_content_tokens(text, tokenizer):
    # No truncation: count exact token IDs using the loaded SD2.1 tokenizer.
    return len(tokenizer.encode(text, add_special_tokens=False, truncation=False))

def expand_ti_tokens(text, pipeline):
    # Most TIs are one token; multi-vector TIs can expand to several IDs.
    # This is the documented Diffusers expansion used by Compel's TI manager.
    return pipeline.maybe_convert_prompt(text, pipeline.tokenizer)

def validate_prompt(text, label, pipeline, limit=MAX_CONTENT_TOKENS):
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f'{label}: empty/invalid prompt')
    raw_count = count_content_tokens(text, pipeline.tokenizer)
    expanded_count = count_content_tokens(expand_ti_tokens(text, pipeline), pipeline.tokenizer)
    effective_count = max(raw_count, expanded_count)
    if effective_count > limit:
        raise ValueError(
            f'{label}: {effective_count} content tokens (raw={raw_count}, '
            f'TI-expanded={expanded_count}); max allowed={limit}. '
            'Shorten the prompt; no truncation will occur.')
    return {'raw':raw_count, 'expanded':expanded_count,
            'effective':effective_count,
            'chunks_needed':max(1, math.ceil(effective_count / CONTENT_PER_CHUNK))}

# Eagerly validate ALL prompts, including the cystic token and negative prompt.
# Any failure stops BEFORE the image generation loop.
negative_stats = validate_prompt(NEGATIVE_PROMPT, 'NEGATIVE_PROMPT', pipe)
prompt_stats = {}
for entry in PROMPTS:
    text = prompt_text(entry)
    stats = validate_prompt(text, entry['id'], pipe)
    prompt_stats[entry['id']] = stats
    print(f"{entry['id']}: {stats['effective']}/{MAX_CONTENT_TOKENS} content tokens; "
          f"{stats['chunks_needed']}/{MAX_CLIP_CHUNKS} chunks")
print('Negative prompt:',negative_stats['effective'],'content tokens')

compel_encoder = Compel(
    tokenizer=pipe.tokenizer,
    text_encoder=pipe.text_encoder,
    textual_inversion_manager=DiffusersTextualInversionManager(pipe),
    truncate_long_prompts=False,
    split_long_text_mode=SplitLongTextMode.WORDS,
    device='cuda',
)

def encode_conditioning(positive_text, negative_text, pipeline, encoder):
    # Tokens were already checked at the top, but validate again for reuse.
    validate_prompt(positive_text, 'positive', pipeline)
    validate_prompt(negative_text, 'negative', pipeline)
    with torch.no_grad():
        positive = encoder.build_conditioning_tensor(positive_text)
        negative = encoder.build_conditioning_tensor(negative_text)
        positive, negative = encoder.pad_conditioning_tensors_to_same_length(
            [positive, negative])
    if positive.ndim != 3 or negative.ndim != 3 or positive.shape != negative.shape:
        raise RuntimeError('Compel positive/negative embeddings must have equal [B,T,D] shape')
    if positive.shape[0] != 1:
        raise RuntimeError('Expected one prompt per call')
    if positive.shape[1] > MAX_CLIP_CHUNKS * CLIP_WINDOW:
        raise ValueError(
            f'Compel needed {positive.shape[1]} positions (> {MAX_CLIP_CHUNKS} '
            f'chunks of {CLIP_WINDOW}). Shorten or simplify the prompt.')
    if positive.shape[1] % CLIP_WINDOW != 0:
        raise RuntimeError('Compel did not return whole SD2.1 CLIP chunks')
    # Make sure the pipeline's UNet sees the dtype/device it expects.
    return (positive.to(device=pipeline.device, dtype=pipeline.unet.dtype),
            negative.to(device=pipeline.device, dtype=pipeline.unet.dtype))

# Pre-encode and validate ALL positive/negative embeddings BEFORE generating.
# Reusing embeddings across five seeds avoids running Compel five times per prompt.
embeddings_by_id = {}
for entry in PROMPTS:
    pid = entry['id']
    positive, negative = encode_conditioning(
        prompt_text(entry), entry.get('negative_prompt',NEGATIVE_PROMPT), pipe, compel_encoder)
    embeddings_by_id[pid] = (positive, negative)
    print(f"Conditioning {pid}: shape={tuple(positive.shape)}; "
          f"chunks={positive.shape[1]//CLIP_WINDOW}; dtype={positive.dtype}")
print('All prompts and Compel embeddings validated. No text was truncated.')


## 6. Generate five images per prompt with reusable Compel embeddings
Each image uses deterministic seeds; images share the same text conditioning for a given prompt. **The pipeline is called with `prompt_embeds` and `negative_prompt_embeds`, not raw `prompt`/`negative_prompt` strings.** This bypasses the regular 77-token tokenization path and preserves both chunks. The tokenizer and inference configuration are saved into metadata.

In [ ]:
import re

def safe_id(value):
    return re.sub(r'[^a-zA-Z0-9_-]+','_',value).strip('_')[:70] or 'prompt'

metadata = []
start = time.time()
for i, entry in enumerate(PROMPTS):
    text_prompt = prompt_text(entry)
    positive_embeds, negative_embeds = embeddings_by_id[entry['id']]
    folder = OUT_DIR / safe_id(entry['id'])
    folder.mkdir(parents=True, exist_ok=True)
    examples = []
    for j in range(IMAGES_PER_PROMPT):
        seed = BASE_SEED + i*1000 + j
        generator = torch.Generator(device='cuda').manual_seed(seed)
        with torch.inference_mode():
            image = pipe(
                prompt_embeds=positive_embeds, negative_prompt_embeds=negative_embeds,
                height=HEIGHT, width=WIDTH, num_inference_steps=STEPS,
                guidance_scale=GUIDANCE, generator=generator,
            ).images[0]
        filename = folder / f'{safe_id(entry["id"])}_seed{seed}.png'
        image.save(filename)
        examples.append(image)
        metadata.append({
            'file': str(filename.relative_to(OUT_DIR)), 'prompt_id': entry['id'],
            'subtype_requested': entry.get('subtype','unspecified'),
            'prompt': text_prompt, 'negative_prompt': entry.get('negative_prompt',NEGATIVE_PROMPT),
            'seed': seed, 'base_model': BASE_MODEL, 'base_commit': base_commit,
            'textual_inversion': TI_MODEL, 'ti_commit': adapter_commit,
            'ti_token_used': bool(entry.get('use_cystic_token',False)),
            'content_tokens': prompt_stats[entry['id']]['effective'],
            'negative_content_tokens':len(pipe.tokenizer(entry.get('negative_prompt',NEGATIVE_PROMPT),add_special_tokens=False,truncation=False)['input_ids']),
            'conditioning_chunks': positive_embeds.shape[1] // CLIP_WINDOW,
            'compel_version': '2.3.1', 'truncate_long_prompts': False,
            'lora_source': LORA_SOURCE or None, 'lora_scale': LORA_SCALE if LORA_SOURCE else None,
            'width': WIDTH, 'height': HEIGHT, 'steps': STEPS, 'guidance': GUIDANCE,
            'clinical_review': 'pending',
        })
    # show a quick preview for each prompt (the saved files are below)
    preview = Image.new('RGB', (WIDTH*len(examples), HEIGHT))
    for j, img in enumerate(examples): preview.paste(img,(j*WIDTH,0))
    display(preview.resize((min(1200,WIDTH*len(examples)),int(HEIGHT*min(1200,WIDTH*len(examples))/(WIDTH*len(examples))))))

with (OUT_DIR/'metadata.jsonl').open('w',encoding='utf-8') as f:
    for row in metadata: f.write(json.dumps(row,ensure_ascii=False)+'\n')
print('Saved',len(metadata),'images in',OUT_DIR,'in',round((time.time()-start)/60,1),'minutes')

In [ ]:
# Write Kaggle CSV/JSON audit outputs BEFORE packaging the generated images.
summary=generation_manifest(metadata,OUT_DIR/'reports',root=OUT_DIR,class_name='Acne',repo_root=DERMSYNREAL_REPO,prompt_partition=PROMPT_PARTITION,model_id=BASE_MODEL,prompt_type=PROMPT_VARIANT)
append_run_log(OUT_DIR,kind='generation',config={'class':'Acne','prompt_partition':PROMPT_PARTITION,'prompt_type':PROMPT_VARIANT,'prompt_count':len(PROMPTS),'images_per_prompt':IMAGES_PER_PROMPT,'generated':summary['generated_images']},artifacts=[OUT_DIR/'reports/generation_manifest.csv',OUT_DIR/'metadata.jsonl'],repo_root=DERMSYNREAL_REPO)
print('Review draft:',OUT_DIR/'reports/synthetic_pool_review_draft.csv','| Run log:',OUT_DIR/'run_log.jsonl')

## 7. Export ZIP and audit basic output counts
The ZIP can be downloaded through the Kaggle **Output** panel. Do not treat samples as clinically validated without blinded morphological review and near-copy checks against real training data.

In [ ]:
import zipfile
zip_path = Path('/kaggle/working/acne_generated.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file(): zf.write(f,arcname=str(f.relative_to(OUT_DIR)))
assert len(metadata) == len(PROMPTS)*IMAGES_PER_PROMPT
print('Download ZIP:',zip_path,'size MB:',round(zip_path.stat().st_size/(1024**2),2))
print('Review metadata at:', OUT_DIR/'metadata.jsonl')

## Reference / reproducibility notes
- [cgDDI acne-cystic textual inversion](https://huggingface.co/hcarrion/acne-cystic); special token `<acne-cystic-class>`
- [SD2.1 community mirror](https://huggingface.co/sd2-community/stable-diffusion-2-1-base): unofficial; provenance must be independently checked before publishing.
- [Hugging Face Diffusers LoRA guide](https://huggingface.co/docs/diffusers/training/lora)
- **Privacy:** generated faces can reproduce training identities. Run full dedup/embedding nearest-neighbor screening before using generated images for classifier training.
- [Compel 2.3.1](https://pypi.org/project/compel/2.3.1/) and [Compel SD2.1 chunking docs](https://github.com/damian0815/compel) — this notebook deliberately pins version 2.3.1 because newer Compel requires Transformers 5.x, while the remaining notebooks pin Transformers 4.56.2.
- `MAX_CONTENT_TOKENS=150` is the maximum per positive or negative prompt, after optional textual-inversion expansion. It corresponds to at most two 77-position CLIP chunks. All prompts are encoded before generation and then paired with padded negative embeddings.